# End-to-end pipeline + integration check

Runs everything from scratch in order and then the full test suite:

**Bronze → Silver → Gold → Analytics → Monitoring → Tests**

All table names come from `config/settings.py`.

In [ ]:
%pip install pytest

In [ ]:
import os
import runpy
import sys

from pyspark.sql import SparkSession

spark = SparkSession.getActiveSession()

REPO_ROOT = os.path.abspath("..")
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from config.settings import BRONZE_SCHEMA, GOLD_SCHEMA, SILVER_SCHEMA, TARGET_CATALOG

for schema in (BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_CATALOG}.{schema}")


def run_script(path):
    # Bronze/Silver are plain scripts that use a global `spark`.
    print(f"\n>>> {path}")
    runpy.run_path(os.path.join(REPO_ROOT, path), init_globals={"spark": spark})

## 1. Bronze

In [ ]:
run_script("src/bronze/load_customer.py")
run_script("src/bronze/load_orders.py")

## 2. Silver (customer first: orders validates its FK against Silver customer)

In [ ]:
run_script("src/silver/customer.py")
run_script("src/silver/orders.py")

from src.validation.order_customer_reference import validate_order_customer_references

validate_order_customer_references(spark)

## 3. Gold

In [ ]:
from src.gold.build_customer_marketing import build_customer_marketing

build_customer_marketing(spark)

## 4. Business questions

In [ ]:
from src.analysis.activation_rate import calculate_activation_rate
from src.analysis.cohort_retention import calculate_cohort_retention
from src.analysis.new_customers import calculate_new_customers
from src.analysis.repeat_purchase_rate import (
    calculate_repeat_purchase_rate,
    highest_repeat_purchase_segment,
)

print("Q1 activation rate (BUILDING):")
display(calculate_activation_rate(spark, "BUILDING"))

print("Q2 new customers 1996-Q1:")
display(calculate_new_customers(spark))

calculate_repeat_purchase_rate(spark)
segment, rate = highest_repeat_purchase_segment(spark)
print(f"Q3 highest repeat purchase rate: {segment} ({rate:.4%})")

print("Q4 cohort retention (1996):")
display(calculate_cohort_retention(spark))

## 5. Monitoring

In [ ]:
from src.monitoring.marketing_monitoring import active_alerts, build_monitoring

build_monitoring(spark)
display(active_alerts(spark))

## 6. Tests

In [ ]:
import shutil
import pytest

TESTS = ("test_bronze.py", "test_gold.py", "test_cohorts_monitoring.py")

for name in TESTS:
    shutil.copy(f"../tests/{name}", f"/tmp/{name}")

result = pytest.main([*(f"/tmp/{name}" for name in TESTS), "-v", "-p", "no:cacheprovider"])

print("pytest exit code:", result)
assert result == 0, "Integration check failed"